In [ ]:
# 第一个单元格：导入必要的库
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns

# 导入自定义的数据加载模块 - 修改为从改进的数据加载器导入
from data_loader import load_and_prepare_data, analyze_data_distribution, check_standardization

# 设置随机种子以确保结果可重复
torch.manual_seed(666)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(666)
np.random.seed(666)

# 设置设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

In [ ]:
# 第二个单元格：定义模型架构
class DenseModel(nn.Module):
    """模仿原始Keras模型的PyTorch实现"""
    def __init__(self, input_dim, hidden_dim=4096, num_classes=102, dropout_rate=0.5):
        super(DenseModel, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.layer2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.layer3 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.layer4 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.output_layer = nn.Linear(hidden_dim, num_classes)
        
        # 初始化权重
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.output_layer(x)
        return torch.softmax(x, dim=1)

In [ ]:
# 第四个单元格：数据加载和标准化检查
# 设置数据路径
base_dir = "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/processed_data"  # 请根据实际路径修改
format = 'mat'  # 保持与原始代码一致

# 加载数据 - 使用改进的数据加载器
# 选择是创建新的scaler还是使用现有scaler
create_new_scaler = False  # 修改为True如果你想创建新的scaler

train_loader, val_loader, test_loader, feature_dim, num_classes, scaler = load_and_prepare_data(
    base_dir=base_dir, 
    create_new_scaler=create_new_scaler,  # 明确指定是否创建新的scaler
    # 如果要指定具体的scaler，可以使用下面的参数
    # scaler_path="/path/to/specific/scaler.pkl",
    format=format,
    batch_size=128,  # 与原始代码相同的batch_size
    shuffle=True,
    seed=666
)

print(f"特征维度: {feature_dim}")
print(f"类别数量: {num_classes}")

# 检查数据是否已标准化 - 使用改进的数据加载器中的函数
print("\n检查训练集数据标准化:")
train_data = next(iter(train_loader))[0]
is_train_standardized = check_standardization(train_data)
print(f"训练集数据是否已标准化: {is_train_standardized}")

print("\n检查验证集数据标准化:")
val_data = next(iter(val_loader))[0]
is_val_standardized = check_standardization(val_data)
print(f"验证集数据是否已标准化: {is_val_standardized}")

print("\n检查测试集数据标准化:")
test_data = next(iter(test_loader))[0]
is_test_standardized = check_standardization(test_data)
print(f"测试集数据是否已标准化: {is_test_standardized}")

# 可选：分析数据分布
# 如果你想了解更多关于数据分布的信息，可以取消下面的注释
from data_loader import analyze_data_distribution
analysis_results = analyze_data_distribution(train_loader, val_loader, test_loader)

In [ ]:
# 第五个单元格：模型定义、训练函数和评估函数
# 创建模型
model = DenseModel(
    input_dim=feature_dim,
    hidden_dim=4096,  # 与原始模型相同
    num_classes=num_classes,
    dropout_rate=0.5  # 与原始模型相同
).to(device)

print(model)

# 设置优化器和损失函数
# 使用与原始模型相同的学习率
optimizer = optim.Adam(model.parameters(), lr=0.00001, weight_decay=0.00001)  # L2正则化
criterion = nn.CrossEntropyLoss()

# 训练函数
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # 使用tqdm显示进度条
    pbar = tqdm(dataloader, desc="Training")
    
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)
        
        # 梯度清零
        optimizer.zero_grad()
        
        # 前向传播
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        # 反向传播和优化
        loss.backward()
        optimizer.step()
        
        # 统计
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
        # 更新进度条
        pbar.set_postfix({
            'loss': running_loss / (pbar.n + 1),
            'acc': 100. * correct / total
        })
    
    return running_loss / len(dataloader), 100. * correct / total

# 评估函数
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # 禁用梯度计算
    with torch.no_grad():
        for inputs, targets in tqdm(dataloader, desc="Evaluation"):
            inputs, targets = inputs.to(device), targets.to(device)
            
            # 前向传播
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            # 统计
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    return running_loss / len(dataloader), 100. * correct / total

In [ ]:
# 第四个单元格：数据加载和标准化检查
# 设置数据路径
base_dir = "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/processed_data"  # 请根据实际路径修改
format = 'mat'  # 保持与原始代码一致

# 加载数据
train_loader, val_loader, test_loader, feature_dim, num_classes, scaler = load_and_prepare_data(
    base_dir=base_dir, 
    format=format,
    batch_size=128,  # 与原始代码相同的batch_size
    shuffle=True,
    seed=666
)

print(f"特征维度: {feature_dim}")
print(f"类别数量: {num_classes}")

# 检查数据是否已标准化
print("\n检查训练集数据标准化:")
train_data = next(iter(train_loader))[0]
is_train_standardized = check_standardization(train_data)
print(f"训练集数据是否已标准化: {is_train_standardized}")

print("\n检查验证集数据标准化:")
val_data = next(iter(val_loader))[0]
is_val_standardized = check_standardization(val_data)
print(f"验证集数据是否已标准化: {is_val_standardized}")

print("\n检查测试集数据标准化:")
test_data = next(iter(test_loader))[0]
is_test_standardized = check_standardization(test_data)
print(f"测试集数据是否已标准化: {is_test_standardized}")

In [ ]:
# 第六个单元格：训练模型
# 训练参数
num_epochs = 25  # 与原始模型相同
best_val_acc = 0.0
train_losses = []
train_accs = []
val_losses = []
val_accs = []

# 训练循环
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    
    # 训练阶段
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # 验证阶段
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    
    # 保存最佳模型
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        print(f"Saving best model with validation accuracy: {val_acc:.2f}%")
        torch.save(model.state_dict(), 'dense_4x4096_model_pytorch.pth')

# 绘制训练过程
plt.figure(figsize=(12, 5))

# 损失曲线
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()

# 准确率曲线
plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Acc')
plt.plot(val_accs, label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Training and Validation Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 第五个单元格：模型定义、训练函数和评估函数
# 创建模型
model = DenseModel(
    input_dim=feature_dim,
    hidden_dim=4096,  # 与原始模型相同
    num_classes=num_classes,
    dropout_rate=0.5  # 与原始模型相同
).to(device)

print(model)

# 设置优化器和损失函数
# 使用与原始模型相同的学习率
optimizer = optim.Adam(model.parameters(), lr=0.00001, weight_decay=0.00001)  # L2正则化
criterion = nn.CrossEntropyLoss()

# 训练函数
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # 使用tqdm显示进度条
    pbar = tqdm(dataloader, desc="Training")
    
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)
        
        # 梯度清零
        optimizer.zero_grad()
        
        # 前向传播
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        # 反向传播和优化
        loss.backward()
        optimizer.step()
        
        # 统计
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
        # 更新进度条
        pbar.set_postfix({
            'loss': running_loss / (pbar.n + 1),
            'acc': 100. * correct / total
        })
    
    return running_loss / len(dataloader), 100. * correct / total

# 评估函数
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # 禁用梯度计算
    with torch.no_grad():
        for inputs, targets in tqdm(dataloader, desc="Evaluation"):
            inputs, targets = inputs.to(device), targets.to(device)
            
            # 前向传播
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            # 统计
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    return running_loss / len(dataloader), 100. * correct / total

In [ ]:
# 第八个单元格：保存模型推理函数，可用于未来预测
def predict_with_model(model_path, input_data, scaler_path=None, device='cuda'):
    """
    使用保存的模型进行预测
    
    参数:
        model_path: 模型权重文件路径
        input_data: 输入数据 [n_samples, n_features]，未标准化
        scaler_path: scaler路径，如果需要标准化数据
        device: 使用的设备，'cuda'或'cpu'
        
    返回:
        预测的类别概率 [n_samples, n_classes]
    """
    # 如果提供了scaler路径，应用标准化
    if scaler_path is not None:
        from data_loader import load_scaler
        try:
            scaler = load_scaler(scaler_path)
            print(f"使用scaler: {scaler_path}")
            input_data = scaler.transform(input_data)
        except Exception as e:
            print(f"加载scaler失败: {e}")
            print("继续使用未标准化的数据...")
    
    # 转换为torch张量
    if isinstance(input_data, np.ndarray):
        input_data = torch.FloatTensor(input_data)
    
    # 加载模型
    feature_dim = input_data.shape[1]
    model = DenseModel(input_dim=feature_dim, hidden_dim=4096, num_classes=102)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    
    # 预测
    with torch.no_grad():
        input_tensor = input_data.to(device)
        predictions = model(input_tensor)
    
    return predictions.cpu().numpy()

# 示例：如何使用上面的函数进行预测
"""
# 1. 在创建新scaler的情况下 - 需要将数据和scaler一起保存
# 先创建和训练模型
train_loader, val_loader, test_loader, feature_dim, num_classes, scaler = load_and_prepare_data(
    base_dir='/path/to/data',
    create_new_scaler=True  # 创建新的scaler
)
# scaler会被保存到 /path/to/data/scalers/ 目录下

# 2. 稍后使用模型和scaler进行预测
import numpy as np
# 加载新的未标准化数据
new_data = np.random.randn(100, 341)  # 示例数据

# 使用模型进行预测，自动应用标准化
predictions = predict_with_model(
    model_path='dense_4x4096_model_pytorch.pth',
    input_data=new_data,
    scaler_path='/path/to/data/scalers/brain_voxel_scaler_latest.pkl'  # 使用latest版本
)

# 获取预测的类别
predicted_classes = np.argmax(predictions, axis=1)

In [ ]:
# 第九个单元格：可视化模型的特征重要性
def compute_feature_importance(model, test_loader, device, num_features=341):
    """
    计算模型的特征重要性，通过对输入特征进行扰动并观察输出变化
    
    这是一个简单的替代方法，用于替代原来的Keras-vis saliency可视化
    """
    model.eval()
    feature_importance = np.zeros(num_features)
    
    # 收集一定数量的样本
    inputs_list = []
    preds_list = []
    
    with torch.no_grad():
        for inputs, _ in tqdm(test_loader, desc="收集样本"):
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = outputs.max(1)
            
            inputs_list.append(inputs.cpu().numpy())
            preds_list.append(preds.cpu().numpy())
            
            # 限制样本数量
            if len(inputs_list) >= 10:  # 只用10个批次的数据
                break
    
    inputs = np.vstack(inputs_list)
    preds = np.concatenate(preds_list)
    
    # 选择一些用于分析的样本索引
    indices_to_visualize = [10, 100, 1000, 2000, 3000]
    
    for idx in indices_to_visualize:
        if idx >= len(inputs):
            continue
            
        input_sample = torch.FloatTensor(inputs[idx:idx+1]).to(device)
        pred_class = preds[idx]
        
        print(f"分析样本 #{idx}, 预测类别: {pred_class}")
        
        # 获取原始预测
        with torch.no_grad():
            original_output = model(input_sample)
            original_prob = original_output[0, pred_class].item()
        
        # 对每个特征进行扰动
        importance = np.zeros(num_features)
        for feat_idx in tqdm(range(num_features), desc=f"扰动特征 (样本 #{idx})"):
            # 创建扰动输入
            perturbed_input = input_sample.clone()
            perturbed_input[0, feat_idx] = 0  # 将特征值设为0
            
            # 获取扰动后的预测
            with torch.no_grad():
                perturbed_output = model(perturbed_input)
                perturbed_prob = perturbed_output[0, pred_class].item()
            
            # 计算影响
            importance[feat_idx] = original_prob - perturbed_prob
        
        # 可视化
        plt.figure(figsize=(12, 6))
        plt.title(f"样本 #{idx}, 预测类别: {pred_class}")
        plt.bar(range(num_features), importance)
        plt.axvspan(0, 15, color='gray', alpha=0.3)
        plt.axvspan(225, 230, color='gray', alpha=0.2)
        plt.xlabel('特征索引')
        plt.ylabel('重要性')
        plt.show()
        
        # 累加到全局特征重要性
        feature_importance += np.abs(importance)
    
    # 归一化全局特征重要性
    feature_importance = feature_importance / len(indices_to_visualize)
    
    # 可视化全局特征重要性
    plt.figure(figsize=(12, 6))
    plt.title("全局特征重要性")
    plt.bar(range(num_features), feature_importance)
    plt.axvspan(0, 15, color='gray', alpha=0.3)
    plt.axvspan(225, 230, color='gray', alpha=0.2)
    plt.xlabel('特征索引')
    plt.ylabel('平均重要性')
    plt.show()
    
    return feature_importance

# 可以在最佳模型上运行此函数
# feature_importance = compute_feature_importance(model, test_loader, device)